## Module submission header
### Submission preparation instructions 
_Completion of this header is mandatory, subject to a 2-point deduction to the assignment._ Only add plain text in the designated areas, i.e., replacing the relevant 'NA's. You must fill out all group member Names and Drexel email addresses in the below markdown list, under header __Module submission group__. It is required to fill out descriptive notes pertaining to any tutoring support received in the completion of this submission under the __Additional submission comments__ section at the bottom of the header. If no tutoring support was received, leave NA in place. You may as well list other optional comments pertaining to the submission at bottom. _Any distruption of this header's formatting will make your group liable to the 2-point deduction._

### Module submission group
- Group member 1
    - Name: Kahf Hussain
    - Email: knh66@drexel.edu
- Group member 2
    - Name: NA
    - Email: NA
- Group member 3
    - Name: NA
    - Email: NA
- Group member 4
    - Name: NA
    - Email: NA

### Additional submission comments
- Tutoring support received: NA
- Other (other): NA

# Assignment group 1: Textual feature extraction and numerical comparison

## Module C _(35 points)_ Similarity of word usage across a document

Here we'll be building up some code to discover how different terms are utilized similarly across a document. For this, our first task will be to create a word frequency counting function.

__C1.__ _(12 points)_ Define a function called `count_words(paragraph, pos = True, lemma = True)` that `return`s a `Counter()` called `frequency`. In `frequency`, each key will consist of a `heading = (text, tag)`, where `text` contains the `word.text` attribute from `spacy` if `lemma = False`, and `word.lemma_` attribute if `True`. Similarly, `tag` should be left empty as `""` if `pos = False` and otherwise contain `word.pos_`. The `Counter()` should simply contain the number of times each `heading` is observed in the `paragraph`.

In [13]:
# C1:Function(12/12)
from collections import Counter
import spacy

nlp = spacy.load("en_core_web_sm")

def count_words(paragraph, pos = True, lemma = True):

    #---Your code starts here

    doc = nlp(paragraph) # Process the paragraph using spaCy
    frequency = Counter() # Initialize a Counter to store frequencies

    for token in doc:
        if token.is_alpha: # Check if the lemma is alphabetic
            text = token.lemma_ if lemma else token.text # Use lemma if specified
            tag = token.pos_ if pos else "" # Include POS if specified
            heading = (text, tag) # Create heading as key
            frequency[heading] += 1 # Update frequency for the heading

    #---Your code ends here
    
    return frequency

Let's make sure your function works by testing it on a short sentence. 

In [14]:
# C1:SanityCheck
count_words("The quick brown fox jumps over the lazy dog.")

Counter({('the', 'DET'): 2,
         ('quick', 'ADJ'): 1,
         ('brown', 'ADJ'): 1,
         ('fox', 'NOUN'): 1,
         ('jump', 'VERB'): 1,
         ('over', 'ADP'): 1,
         ('lazy', 'ADJ'): 1,
         ('dog', 'NOUN'): 1})

__C2.__ _(8 pts)_ Next, define a function called `book_TDM(book_id, pos = True, lemma = True)` and copy into it the TDM-producing code from __Section 2.1.5.1__ of the lecture notes, now `return`-ing `TDM` and `all_words`. Once copied, modify this function to call `count_words` appropriately, now passing through the user of `book_TDM`'s specified `lemma` and `pos` arguments.

In [15]:
# C2:Function(8/8)
import numpy as np
from collections import Counter
import re

def book_TDM(book_id, pos = True, lemma = True):

    #---Your code starts here---

    # Read the book text
    with open(f"./data/books/{book_id}.txt", "r", encoding="utf-8") as file:
        text = file.read()

    # Split text into sentences
    sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?)\s', text) # Added more to regex from Exercise

    #Initialize master set of all words and document frequencies
    all_words = set()
    all_doc_frequencies = {}

    # Loop over sentences to calculate frequencies
    for j, sentence in enumerate(sentences):
        frequency = count_words(sentence, pos = pos, lemma = lemma) # Call count_words
        all_doc_frequencies[j] = frequency # Store the frequency for each sentence
        doc_words = set(frequency.keys()) # Get unique words from the sentence
        all_words = all_words.union(doc_words) # Add to the master set of words

    # Create teh TDM
    TDM = np.zeros((len(all_words), len(all_doc_frequencies)))

    # Ordering for rows alphabetically
    all_words = sorted(list(all_words))

    # Populate the TDM
    for j in all_doc_frequencies:
        for i, word in enumerate(all_words):
            TDM[i, j] = all_doc_frequencies[j][word]

    #---Your code ends here---

    return TDM, all_words


To test your code's function, let's process `book_id = 84` with both of `pos = True` and `lemma = True` and print out the `TDM`'s `.shape` attribute and the first ten terms in `all_words`.

In [16]:
# C2:SanityCheck

TDM, terms = book_TDM("84", pos = True, lemma = True)
terms[:10]

[('Abbey', 'PROPN'),
 ('Adam', 'PROPN'),
 ('Adieu', 'PROPN'),
 ('Africa', 'PROPN'),
 ('Agatha', 'PROPN'),
 ('Agrippa', 'PROPN'),
 ('Al', 'PROPN'),
 ('Alas', 'PROPN'),
 ('Albertus', 'PROPN'),
 ('Alphonse', 'PROPN')]

In [17]:
# C2:SanityCheck

TDM.shape

(6162, 2890)

__C3.__ _(8 pts)_ Next, your job is to define two functions. The first is `sim(u,v)`, which shoud take two arbitrary numeric vectors and compute/output the `cosine_similarity`, as described in __Section 1.1.2.10__.  

The second function is `term_sims(i, TDM)`, which should utilize the first function (`sim` function) to output a list of cosine similarity values (`sim_values`) between the word/row `i` and all others (rows) in the `TDM`.

Note: each of these functions can be straightforwardly completed using a single line of code! Exhibit your knowledge of comprehensions and vectorization!

In [18]:
# C3:Function(4/8)
def sim(u, v):
    
    #---Your code starts here
    cosine_similarity = np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))
    #---Your code ends here
    
    return cosine_similarity

In [19]:
# C3:SanityCheck

print("Exactly similar:", sim(np.array([1,2,3]), np.array([1,2,3])))
print("Exactly dissimilar:", sim(np.array([1,2,3]), np.array([-1,-2,-3])))
print("In the middle:", sim(np.array([1,1]), np.array([-1,1])))

Exactly similar: 1.0
Exactly dissimilar: -1.0
In the middle: 0.0


In [20]:
# C3:Function(4/8)

def term_sims(i, TDM):
    
    #---Your code starts here

    sim_values = [sim(TDM[i], TDM[j]) for j in range(TDM.shape[0])]

    #---Your code ends here
    
    return sim_values

In [21]:
# C3:SanityCheck

# Compare word/row 0 to all other (rows) in the TDM
term_sims(0, TDM)

[np.float64(1.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float6

__C4.__ _(7 pts)_ Finally, your goal now is to a write function, `most_similar(term, terms, TDM, top = 25)`, that utilizes `term_sims` to output a sorted list of the `top = N` terms (`top_n_terms`) most similar to one specified (`term`). The output data type should be a list of lists, with each inner list representing information for a similar term as: `[row_ix, similarity, term]`. 

\[Hint: to locate the row containing the term of interest, utilize the list `.index()` method in application to the `terms` argument.\]

In [22]:
# C4:Function(6/7)

def most_similar(term, terms, TDM, top=25):
    
    #---Your code starts here---

    # Find the row index of the specified term
    row_index = terms.index(term)

    # Compute cosine similarity values for the specified term's row
    similarities = term_sims(row_index, TDM)

    # Pair similarities with row indices and terms
    similar_terms = [
        [ix, sim, terms[ix]]
        for ix, sim in enumerate(similarities)
        if ix != row_index
    ]

    # Sort terms by similarity in descending order
    similar_terms = sorted(similar_terms, key = lambda x: x[1], reverse = True)

    # Select the top n terms
    top_n_terms = similar_terms[:top]

    #---Your code ends here---
    
    return top_n_terms

Now, let's test your functions utility on a `TDM` produced for `book_id = 84` and exhibit the top 25 similar terms to both of `('monster', 'NOUN')` and `('beautiful', 'ADJ')`.

In [23]:
# C4:SanityCheck

most_similar(('monster', 'NOUN'), terms, TDM, top = 25)

[[3418, np.float64(0.1765469659009499), ('let', 'VERB')],
 [364, np.float64(0.17407765595569785), ('abhorred', 'ADJ')],
 [413, np.float64(0.17407765595569785), ('accursed', 'NOUN')],
 [452, np.float64(0.17407765595569785), ('adjuration', 'NOUN')],
 [727, np.float64(0.17407765595569785), ('asseveration', 'NOUN')],
 [808, np.float64(0.17407765595569785), ('awe', 'NOUN')],
 [922, np.float64(0.17407765595569785), ('besiege', 'VERB')],
 [946, np.float64(0.17407765595569785), ('blameless', 'ADJ')],
 [1005, np.float64(0.17407765595569785), ('boy', 'INTJ')],
 [1210, np.float64(0.17407765595569785), ('choke', 'VERB')],
 [1469, np.float64(0.17407765595569785), ('convulse', 'VERB')],
 [1585, np.float64(0.17407765595569785), ('dark', 'NOUN')],
 [1865, np.float64(0.17407765595569785), ('disown', 'VERB')],
 [2398, np.float64(0.17407765595569785), ('fiend', 'INTJ')],
 [2400, np.float64(0.17407765595569785), ('fiend', 'VERB')],
 [2500, np.float64(0.17407765595569785), ('forehead', 'NOUN')],
 [2588, np

In [24]:
# C4:SanityCheck

most_similar(('beautiful', 'ADJ'), terms, TDM, top = 25)

[[4343, np.float64(0.26726124191242434), ('priest', 'NOUN')],
 [4665, np.float64(0.26726124191242434), ('resemblance', 'NOUN')],
 [5399, np.float64(0.26726124191242434), ('temp', 'ADJ')],
 [2906, np.float64(0.2182178902359924), ('horrid', 'ADJ')],
 [3354, np.float64(0.19738550848793068), ('lake', 'NOUN')],
 [172, np.float64(0.1889822365046136), ('La', 'PROPN')],
 [176, np.float64(0.1889822365046136), ('Lavenza', 'PROPN')],
 [219, np.float64(0.1889822365046136), ('Montalegre', 'PROPN')],
 [254, np.float64(0.1889822365046136), ('Pays', 'PROPN')],
 [335, np.float64(0.1889822365046136), ('Uri', 'PROPN')],
 [336, np.float64(0.1889822365046136), ('Valais', 'PROPN')],
 [337, np.float64(0.1889822365046136), ('Vaud', 'PROPN')],
 [340, np.float64(0.1889822365046136), ('Villa', 'PROPN')],
 [498, np.float64(0.1889822365046136), ('affright', 'NOUN')],
 [555, np.float64(0.1889822365046136), ('alluring', 'ADJ')],
 [581, np.float64(0.1889822365046136), ('amid', 'ADP')],
 [777, np.float64(0.18898223650

In [26]:
# C4:Inline

# Comment on the ordered results returned in the sanity checks.

# They make sense to me. 'Let' is a verb that can be often used to describe the 'monster's' action. The following words are adj. and Nouns that closely relate to words that we would associate with monsters.

# In the second example, 'beautiful' shows strong connection with 'priest' which don't make as much sense. But terms like 'resemblance', 'horrid', 'lake' make sense. As they are pretty close to 'beautiful' in terms of linguistic usage.

# Do you think the algorithm is exhibiting sensible results? print "Yes" or "No"
print("Yes")

Yes
